# 00 Setup and exploration

Notebook 00 establishes the working environment and performs the first structured exploration of the GA4 public ecommerce export. It is kept as exploratory evidence for the final submission, while the reusable table building pipeline starts in Notebook 01.

---

## Purpose

- Confirm that Colab, Google Drive, and BigQuery are correctly connected
- Define the GA4 source table, selected `_TABLE_SUFFIX` window, and exploratory sample size
- Save `run_metadata.json` as record of the source window used for this exploration
- Inspect the raw GA4 schema, event mix, `event_params`, item arrays, and candidate segmentation fields

# 1.0 Data imports and environment setup


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Install packages
#------------------------------------------------------------------------------
# Install the BigQuery and Arrow packages used for data access
%pip -q install pandas-gbq google-cloud-bigquery pyarrow


In [ ]:
#------------------------------------------------------------------------------
# 1.2 Import libraries and notebook display settings
#------------------------------------------------------------------------------
from pathlib import Path
import json
import pandas as pd, pandas_gbq
from IPython.display import display

# Widen pandas displays so nested GA4 fields can be inspected
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 160)


In [ ]:
#------------------------------------------------------------------------------
# 1.4 Authenticate to Google Cloud and mount Drive
#------------------------------------------------------------------------------
from google.colab import auth, drive
import google.auth

auth.authenticate_user()
CREDENTIALS, _ = google.auth.default()
pandas_gbq.context.credentials = CREDENTIALS

drive.mount("/content/drive", force_remount=True)



Mounted at /content/drive


# 2.0 Global config and run metadata

The project configuration is recorded near the start of the notebook so the source table, date window, sample size, and output paths are clear before any exploration is run. The only saved file from this setup step is metadata record for reproducibility.

**Process**:
1. Define the BigQuery project, public GA4 wildcard table, and selected `_TABLE_SUFFIX` date range.
2. Create the main Drive output folder used by the capstone project.
3. Save the run metadata.
4. Keep the remaining checks in this notebook as displayed EDA evidence only.

In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core config and helpers
#------------------------------------------------------------------------------
# Set the billing project used for BigQuery queries
BQ_PROJECT_ID = "ga4-project-490603"
pandas_gbq.context.project = BQ_PROJECT_ID

# Keep BigQuery reads in the US location used by the public dataset and set link
pandas_gbq.context.location = "US"
GA4_PUBLIC_DATASET_TABLE = "bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*"
USE_BQSTORAGE_API = False

# Final project window and small raw sample used for exploratory previews
START_DATE = "20201101"
END_DATE = "20210228"
RAW_SAMPLE_LIMIT = 1000

# Build the Drive output paths
CAPSTONE_DIR = Path("/content/drive/MyDrive/Capstone_Project")
OUTPUT_DIR = CAPSTONE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_METADATA_PATH = OUTPUT_DIR / "run_metadata.json"

# Print a clear banner before each major check
def print_banner(title):
    print("\n" + "-" * 78); print(title); print("-" * 78)

# Overwrite small JSON outputs so reruns stay clean
def save_json_overwrite(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

# Run BigQuery SQL and return a pandas DataFrame
def run_bq(query, project_id=BQ_PROJECT_ID):
    return pandas_gbq.read_gbq(query, project_id=project_id, dialect="standard", use_bqstorage_api=USE_BQSTORAGE_API)

# Save only the metadata required by the next notebook
run_metadata = {"notebook": "00_setup_and_exploration", "START_DATE": START_DATE, "END_DATE": END_DATE,
                "bq_project_id": BQ_PROJECT_ID, "source_table": GA4_PUBLIC_DATASET_TABLE,
                "raw_sample_limit": RAW_SAMPLE_LIMIT, "capstone_dir": str(CAPSTONE_DIR)}
save_json_overwrite(RUN_METADATA_PATH, run_metadata)

print("Saved run metadata ->", RUN_METADATA_PATH)
display(pd.DataFrame([run_metadata]))


Saved run metadata -> /content/drive/MyDrive/Capstone_Project/outputs/run_metadata.json


,notebook,START_DATE,END_DATE,bq_project_id,source_table,raw_sample_limit,capstone_dir
0,00_setup_and_exploration,20201101,20210228,ga4-project-490603,bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*,1000,/content/drive/MyDrive/Capstone_Project


# 3.0 Raw sample and schema inspection

A bounded raw sample gives a practical first view of the GA4 export before any transformation logic is applied. The sample helps confirm which columns are nested, which fields are repeated, and how event-level records appear in pandas, while still keeping local memory use controlled.

**Process**:
1. Pull a small `SELECT *` sample from the configured GA4 suffix window.
2. Display the raw sample shape and first rows for schema inspection.
3. Review sample level column types, null rates, event names, and always-null columns.
4. Keep these checks as exploratory notebook evidence rather than saved dashboard outputs.


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Pull raw GA4 sample
#------------------------------------------------------------------------------
print_banner("3.1 Pull raw GA4 sample")

# Build a bounded raw sample query from the configured suffix window
query_raw = f"""
SELECT *
FROM `{GA4_PUBLIC_DATASET_TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}'
LIMIT {RAW_SAMPLE_LIMIT}
"""

df_ga4_raw = run_bq(query_raw)

print("Loaded raw sample:", df_ga4_raw.shape)
display(df_ga4_raw.head(10))



------------------------------------------------------------------------------
3.1 Pull raw GA4 sample
------------------------------------------------------------------------------
Downloading: 100%|██████████|
Loaded raw sample: (1000, 23)


,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,privacy_info,user_properties,user_first_touch_timestamp,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items
0,20210101,1609468318563156,first_visit,"[{'key': 'ga_session_number', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_id', 'value':...",<NA>,NaN,1177212143,<NA>,None,1003046.9452926974,"{'analytics_storage': None, 'ads_storage': None, 'uses_transient_token': 'No'}",[],1609468318563156,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': 'Microsoft', 'mobile_model_name': '<Other>', 'mobile_marketing_name': '<Other>', 'mobile_os_hardware_model': No...","{'continent': 'Europe', 'sub_continent': 'Western Europe', 'country': 'France', 'region': 'Auvergne-Rhone-Alpes', 'city': '(not set)', 'metro': '(not set)'}",None,"{'medium': 'organic', 'name': '(organic)', 'source': 'google'}",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[]
1,20210101,1609468318563156,page_view,"[{'key': 'page_location', 'value': {'string_value': 'https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Google+Dino+Game+Tee', 'int_value': Non...",<NA>,NaN,1177212143,<NA>,None,1003046.9452926974,"{'analytics_storage': None, 'ads_storage': None, 'uses_transient_token': 'No'}",[],1609468318563156,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': 'Microsoft', 'mobile_model_name': '<Other>', 'mobile_marketing_name': '<Other>', 'mobile_os_hardware_model': No...","{'continent': 'Europe', 'sub_continent': 'Western Europe', 'country': 'France', 'region': 'Auvergne-Rhone-Alpes', 'city': '(not set)', 'metro': '(not set)'}",None,"{'medium': 'organic', 'name': '(organic)', 'source': 'google'}",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[]
2,20210101,1609468318563156,session_start,"[{'key': 'ga_session_number', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'engaged_session_event', ...",<NA>,NaN,1177212143,<NA>,None,1003046.9452926974,"{'analytics_storage': None, 'ads_storage': None, 'uses_transient_token': 'No'}",[],1609468318563156,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': 'Microsoft', 'mobile_model_name': '<Other>', 'mobile_marketing_name': '<Other>', 'mobile_os_hardware_model': No...","{'continent': 'Europe', 'sub_continent': 'Western Europe', 'country': 'France', 'region': 'Auvergne-Rhone-Alpes', 'city': '(not set)', 'metro': '(not set)'}",None,"{'medium': 'organic', 'name': '(organic)', 'source': 'google'}",2100450278,WEB,None,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[]
3,20210101,1609468320458554,user_engagement,"[{'key': 'engaged_session_event', 'value': {'string_value': None, 'int_value': 1, 'float_value': None, 'double_value': None}}, {'key': 'ga_session_number', ...",<NA>,NaN,7525162155,<NA>,None,1003046.9452926974,"{'analytics_storage': None, 'ads_storage': None, 'uses_transient_token': 'No'}",[],1609468318563156,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': 'Microsoft', 'mobile_model_name': '<Other>', 'mobile_marketing_name': '<Other>', 'mobile_os_hardware_model': No...","{'continent': 'Europe', 'sub_continent': 'Western Europe', 'country': 'France', 'region': 'Auvergne-Rhone-Alpes', 'city': '(not set)', 'metro': '(not set)'}",None,"{'medium': 'organic', 'name': '(organic)', 'source': 'g

In [ ]:
#------------------------------------------------------------------------------
# 3.2 Schema sanity checks
#------------------------------------------------------------------------------
print_banner("3.2 Schema sanity checks")

# Build a quick table of raw column names and pandas dtypes.
schema_preview = pd.DataFrame({"col": df_ga4_raw.columns, "dtype": [str(x) for x in df_ga4_raw.dtypes]})

# Calculate sample null rates for every raw column.
null_rates = df_ga4_raw.isna().mean().sort_values(ascending=False).reset_index()

# Rename columns for clearer display.
null_rates.columns = ["column", "sample_null_rate"]

# Display schema and missingness checks.
display(schema_preview)
display(null_rates.head(25))

# Count event names in the raw sample.
print("Unique event_name values in sample:", df_ga4_raw["event_name"].nunique())
display(df_ga4_raw["event_name"].value_counts().head(20).reset_index(name="n_events"))



------------------------------------------------------------------------------
3.2 Schema sanity checks
------------------------------------------------------------------------------


,col,dtype
0,event_date,object
1,event_timestamp,Int64
2,event_name,object
3,event_params,object
4,event_previous_timestamp,Int64
5,event_value_in_usd,float64
6,event_bundle_sequence_id,Int64
7,event_server_timestamp_offset,Int64
8,user_id,object
9,user_pseudo_id,object


,column,sample_null_rate
0,event_previous_timestamp,1.000
1,event_value_in_usd,1.000
2,event_server_timestamp_offset,1.000
3,app_info,1.000
4,event_dimensions,1.000
5,user_id,1.000
6,user_first_touch_timestamp,0.007
7,event_bundle_sequence_id,0.000
8,event_params,0.000
9,event_timestamp,0.000


Unique event_name values in sample: 10


,event_name,n_events
0,page_view,382
1,user_engagement,265
2,scroll,136
3,session_start,79
4,first_visit,70
5,view_item,49
6,add_shipping_info,9
7,view_promotion,6
8,view_search_results,3
9,add_payment_info,1


In [ ]:
#------------------------------------------------------------------------------
# 3.3 Always null columns in the raw sample
#------------------------------------------------------------------------------
print_banner("3.3 Always-null columns in the raw sample")

# Find columns that are fully null in this sample and store as list
all_null_cols = [c for c in df_ga4_raw.columns if df_ga4_raw[c].isna().all()]
df_all_null = pd.DataFrame({"all_null_column": all_null_cols})

print("All-null columns in this sample:", len(all_null_cols))
display(df_all_null)
print("Displayed only. Development CSV output is not saved in the final notebook.")



------------------------------------------------------------------------------
3.3 Always-null columns in the raw sample
------------------------------------------------------------------------------
All-null columns in this sample: 6


,all_null_column
0,event_previous_timestamp
1,event_value_in_usd
2,event_server_timestamp_offset
3,user_id
4,app_info
5,event_dimensions


Displayed only. Development CSV output is not saved in the final notebook.


# 4.0 Event mix, event_params and item coverage

Aggregate SQL checks provide a more reliable view than the small local sample because they scan the selected GA4 suffix window directly in BigQuery. These checks confirm whether the planned ecommerce funnel is feasible and whether the nested `event_params` and `items` structures contain enough useful information for later diagnostics and modelling.

**Process**:
1. Count the most common event names across the selected date window to confirm the overall event mix.
2. Check whether the required funnel events appear, including product views, cart activity, checkout steps, and purchases.
3. Unnest `event_params` to identify the most frequent parameter keys and confirm the availability of session and engagement fields.
4. Unnest the `items` array to check which events carry product information and whether item-level aggregation is feasible.
5. Display the checks inside the notebook.

In [ ]:
#------------------------------------------------------------------------------
# 4.1 Event mix over the selected date window
#------------------------------------------------------------------------------
print_banner("4.1 Event mix over the selected date window")

# Count event types across the selected BigQuery window
query_event_counts = f"""
SELECT event_name, COUNT(*) AS n_events
FROM `{GA4_PUBLIC_DATASET_TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY event_name
ORDER BY n_events DESC
LIMIT 50
"""

df_event_counts = run_bq(query_event_counts)
display(df_event_counts)



------------------------------------------------------------------------------
4.1 Event mix over the selected date window
------------------------------------------------------------------------------
Downloading: 100%|██████████|


,event_name,n_events
0,page_view,1350428
1,user_engagement,1058721
2,scroll,493072
3,view_item,386068
4,session_start,354970
5,first_visit,257462
6,view_promotion,190104
7,add_to_cart,58543
8,begin_checkout,38757
9,select_item,31007


In [ ]:
#------------------------------------------------------------------------------
# 4.2 event_params key frequency
#------------------------------------------------------------------------------
print_banner("4.2 event_params key frequency")

# Count how often each event_params key appears
query_param_keys = f"""
SELECT ep.key AS param_key, COUNT(*) AS n_rows_with_key
FROM `{GA4_PUBLIC_DATASET_TABLE}` e, UNNEST(e.event_params) AS ep
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY param_key
ORDER BY n_rows_with_key DESC
LIMIT 200
"""

df_param_keys = run_bq(query_param_keys)
display(df_param_keys.head(60))
print("Top-200 param keys returned:", df_param_keys.shape[0])



------------------------------------------------------------------------------
4.2 event_params key frequency
------------------------------------------------------------------------------
Downloading: 100%|██████████|


,param_key,n_rows_with_key
0,ga_session_number,4295584
1,ga_session_id,4295584
2,page_location,4295584
3,page_title,4276104
4,engaged_session_event,4116237
5,session_engaged,3996312
6,debug_mode,3682991
7,page_referrer,3206512
8,all_data,2828174
9,clean_event,2828040


Top-200 param keys returned: 34


In [ ]:
#------------------------------------------------------------------------------
# 4.3 Items coverage by event_name
#------------------------------------------------------------------------------
print_banner("4.3 Items coverage by event_name")

# Measure item-array availability by event type
query_items_coverage = f"""
SELECT event_name, COUNT(*) AS n_events,
       COUNTIF(ARRAY_LENGTH(items) > 0) AS n_events_with_items,
       SAFE_DIVIDE(COUNTIF(ARRAY_LENGTH(items) > 0), COUNT(*)) AS pct_with_items
FROM `{GA4_PUBLIC_DATASET_TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY event_name
ORDER BY n_events DESC
LIMIT 50
"""

df_items_cov = run_bq(query_items_coverage)
display(df_items_cov)



------------------------------------------------------------------------------
4.3 Items coverage by event_name
------------------------------------------------------------------------------
Downloading: 100%|██████████|


,event_name,n_events,n_events_with_items,pct_with_items
0,page_view,1350428,0,0.000000
1,user_engagement,1058721,0,0.000000
2,scroll,493072,0,0.000000
3,view_item,386068,241876,0.626511
4,session_start,354970,0,0.000000
5,first_visit,257462,0,0.000000
6,view_promotion,190104,134602,0.708044
7,add_to_cart,58543,58543,1.000000
8,begin_checkout,38757,31991,0.825425
9,select_item,31007,31006,0.999968


# 5.0 Candidate fields for later gold tables

The exploration now narrows from general schema inspection to the fields that are worth carrying forward. The aim is to separate useful pre-purchase behavioural and contextual inputs from fields that are only appropriate for reporting, audit, or labels.

**Process**:
1. Review candidate `event_params` keys that may support sessionisation, behavioural features, traffic attribution, and page context.
2. Separate modelling  fields from purchase context fields that should not be used as predictive inputs.
3. Inspect selected parameter values in the raw sample to understand whether each field is stored as a string, integer, or numeric value.
4. Results used to keep Notebook 01 focused on interpretable and reusable fields rather than extracting every available parameter.

In [ ]:
#------------------------------------------------------------------------------
# 5.1 Candidate param keys for modelling and reporting
#------------------------------------------------------------------------------
print_banner("5.1 Candidate param keys for modelling and reporting")

# Define likely pre-purchase keys for modelling features
CANDIDATE_KEYS_MODELLING = ["ga_session_id", "ga_session_number", "session_engaged", "engagement_time_msec", "page_location", "page_referrer", "page_title", "source", "medium", "campaign", "term", "content"]

# Define purchase and reporting keys that should not become model inputs
CANDIDATE_KEYS_REPORTING = ["transaction_id", "shipping_tier", "payment_type", "tax", "shipping", "coupon", "currency", "value"]

# Convert available keys into a set
available_param_keys = set(df_param_keys["param_key"].astype(str))
candidate_rows = []

# Check keys
for key in CANDIDATE_KEYS_MODELLING:
    candidate_rows.append({"param_key": key, "key_group": "modelling", "found_in_window": key in available_param_keys})
for key in CANDIDATE_KEYS_REPORTING:
    candidate_rows.append({"param_key": key, "key_group": "reporting", "found_in_window": key in available_param_keys})

df_candidate_keys = pd.DataFrame(candidate_rows)

# Store found keys for later preview
candidate_keys_found_modelling = df_candidate_keys.query("key_group == 'modelling' and found_in_window")["param_key"].tolist()
candidate_keys_found_reporting = df_candidate_keys.query("key_group == 'reporting' and found_in_window")["param_key"].tolist()

display(df_candidate_keys)
print("Modelling keys found:", candidate_keys_found_modelling)
print("Reporting keys found but not used for modelling:", candidate_keys_found_reporting)



------------------------------------------------------------------------------
5.1 Candidate param keys for modelling and reporting
------------------------------------------------------------------------------


,param_key,key_group,found_in_window
0,ga_session_id,modelling,True
1,ga_session_number,modelling,True
2,session_engaged,modelling,True
3,engagement_time_msec,modelling,True
4,page_location,modelling,True
5,page_referrer,modelling,True
6,page_title,modelling,True
7,source,modelling,True
8,medium,modelling,True
9,campaign,modelling,True


Modelling keys found: ['ga_session_id', 'ga_session_number', 'session_engaged', 'engagement_time_msec', 'page_location', 'page_referrer', 'page_title', 'source', 'medium', 'campaign', 'term']
Reporting keys found but not used for modelling: ['transaction_id', 'shipping_tier', 'payment_type', 'tax', 'coupon', 'currency', 'value']


In [ ]:
#------------------------------------------------------------------------------
# 5.2 Preview selected event_params values
#------------------------------------------------------------------------------
print_banner("5.2 Preview selected event_params values")

# Extract one typed GA4 event_param value from a repeated key-value list
def get_event_param_value(event_params_list, key):
    if event_params_list is None:
        return None
    for item in event_params_list:
        if item.get("key") == key:
            value = item.get("value") or {}
            for value_col in ["string_value", "int_value", "double_value", "float_value"]:
                if value.get(value_col) is not None:
                    return value.get(value_col)
    return None

keys_to_preview = candidate_keys_found_modelling[:10]

# Add temporary sample columns for each selected event_param key
for key in keys_to_preview:
    df_ga4_raw[f"ep__{key}"] = df_ga4_raw["event_params"].apply(lambda x: get_event_param_value(x, key))

# Collect temporary preview columns
ep_cols = [c for c in df_ga4_raw.columns if c.startswith("ep__")]

# Summarise how often each selected key appears in the raw sample
df_param_coverage_sample = pd.DataFrame([{"param_key": k, "sample_non_null_rate": float(df_ga4_raw[f'ep__{k}'].notna().mean())} for k in keys_to_preview])

display(df_ga4_raw[ep_cols].head(10))
display(df_param_coverage_sample.sort_values("sample_non_null_rate", ascending=False))



------------------------------------------------------------------------------
5.2 Preview selected event_params values
------------------------------------------------------------------------------


,ep__ga_session_id,ep__ga_session_number,ep__session_engaged,ep__engagement_time_msec,ep__page_location,ep__page_referrer,ep__page_title,ep__source,ep__medium,ep__campaign
0,3209612510,1,1,NaN,https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Google+Dino+Game+Tee,None,Google Dino Game Tee,None,None,None
1,3209612510,1,0,NaN,https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Google+Dino+Game+Tee,None,Google Dino Game Tee,google,organic,(organic)
2,3209612510,1,None,NaN,https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Google+Dino+Game+Tee,None,Google Dino Game Tee,None,None,None
3,3209612510,1,1,2075.0,https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Google+Dino+Game+Tee,None,Google Dino Game Tee,None,None,None
4,3209612510,1,1,9.0,https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Google+Dino+Game+Tee,None,Google Dino Game Tee,None,None,None
5,7473279052,1,1,15567.0,https://shop.googlemerchandisestore.com/Google+Redesign/Campus+Collection/Google+Sunnyvale+Campus+Bottle,https://shop.googlemerchandisestore.com/Google+Redesign/New?,Google Sunnyvale Campus Bottle,None,None,None
6,7473279052,1,1,21900.0,https://shop.googlemerchandisestore.com/Google+Redesign/Lifestyle/Fun/Malibu+Sunglasses.axd,None,The Google Merchandise Store/Malibu Sunglasses,<Other>,<Other>,<Other>
7,7473279052,1,1,7.0,https://shop.googlemerchandisestore.com/,None,Home,None,None,None
8,7473279052,1,1,6603.0,https://shop.googlemerchandisestore.com/Google+Redesign/Accessories/Google+Clear+Framed+Yellow+Shades,https://shop.googlemerchandisestore.com/basket.html?,Google Clear Framed Yellow Shades,<Other>,<Other>,<Other>
9,7473279052,1,1,2187.0,https://shop.googlemerchandisestore.com/Google+Redesign/Campus+Collection/Google+Sunnyvale+Campus+Bottle,https://shop.googlemerchandisestore.com/Google+Redesign/New?,Google Sunnyvale Campus Bottle,None,None,None


,param_key,sample_non_null_rate
0,ga_session_id,1.000
1,ga_session_number,1.000
4,page_location,1.000
6,page_title,0.995
2,session_engaged,0.927
3,engagement_time_msec,0.620
5,page_referrer,0.424
7,source,0.282
8,medium,0.282
9,campaign,0.282


# 6.0 Segmentation field sanity checks

Device, geography, and traffic source fields are central to the later funnel diagnostics, model review, and Power BI dashboard. These checks confirm whether the main segmentation fields are populated clearly enough to support weekly segment analysis and business-readable labels in later notebooks.

**Process**:
1. Inspect device category coverage to confirm whether desktop, mobile, and tablet segments can be analysed.
2. Inspect country and other geography fields to check whether country level funnel diagnostics are feasible.
3. Review source, medium, and campaign-style fields to understand how traffic channels appear in the raw export.
4. Use these checks to support the standardised `channel_source`, `channel_medium`, and `channel_segment` logic created in Notebook 01.

In [ ]:
#------------------------------------------------------------------------------
# 6.1 Segmentation field sanity checks
#------------------------------------------------------------------------------
print_banner("6.1 Segmentation field sanity checks")

# Read nested dict fields
def get_nested_value(d, path, default=None):
    cur = d
    for p in path:
        if cur is None or not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur

# Start the segmentation preview with event name and platform
df_seg = pd.DataFrame({"event_name": df_ga4_raw["event_name"], "platform": df_ga4_raw["platform"]})

# Extract core device fields from the nested device record.
df_seg["device_category"] = df_ga4_raw["device"].apply(lambda x: get_nested_value(x, ["category"]))
df_seg["os"] = df_ga4_raw["device"].apply(lambda x: get_nested_value(x, ["operating_system"]))
df_seg["browser"] = df_ga4_raw["device"].apply(lambda x: get_nested_value(x, ["web_info", "browser"]))

# Extract country and region from the nested geo record
df_seg["country"] = df_ga4_raw["geo"].apply(lambda x: get_nested_value(x, ["country"]))
df_seg["region"] = df_ga4_raw["geo"].apply(lambda x: get_nested_value(x, ["region"]))

# Extract first-touch traffic source fields from the nested traffic source record
df_seg["traffic_source_source"] = df_ga4_raw["traffic_source"].apply(lambda x: get_nested_value(x, ["source"]))
df_seg["traffic_source_medium"] = df_ga4_raw["traffic_source"].apply(lambda x: get_nested_value(x, ["medium"]))

# Calculate missingness
seg_nulls = df_seg.isna().mean().sort_values(ascending=False).reset_index()

seg_nulls.columns = ["field", "sample_null_rate"]

display(df_seg.head(10))
display(seg_nulls)
print("Top device categories:")
display(df_seg["device_category"].value_counts(dropna=False).head(10).reset_index(name="n_events"))
print("Top countries:")
display(df_seg["country"].value_counts(dropna=False).head(10).reset_index(name="n_events"))



------------------------------------------------------------------------------
6.1 Segmentation field sanity checks
------------------------------------------------------------------------------


,event_name,platform,device_category,os,browser,country,region,traffic_source_source,traffic_source_medium
0,first_visit,WEB,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,google,organic
1,page_view,WEB,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,google,organic
2,session_start,WEB,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,google,organic
3,user_engagement,WEB,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,google,organic
4,page_view,WEB,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,google,organic
5,scroll,WEB,mobile,Web,Safari,India,Maharashtra,<Other>,<Other>
6,user_engagement,WEB,mobile,Web,Safari,India,Maharashtra,<Other>,<Other>
7,page_view,WEB,mobile,Web,Safari,India,Maharashtra,<Other>,<Other>
8,user_engagement,WEB,mobile,Web,Safari,India,Maharashtra,<Other>,<Other>
9,user_engagement,WEB,mobile,Web,Safari,India,Maharashtra,<Other>,<Other>


,field,sample_null_rate
0,event_name,0.0
1,platform,0.0
2,device_category,0.0
3,os,0.0
4,browser,0.0
5,country,0.0
6,region,0.0
7,traffic_source_source,0.0
8,traffic_source_medium,0.0


Top device categories:


,device_category,n_events
0,desktop,645
1,mobile,346
2,tablet,9


Top countries:


,country,n_events
0,Portugal,325
1,United States,231
2,India,152
3,Canada,59
4,China,36
5,Poland,23
6,Israel,21
7,United Kingdom,18
8,Singapore,13
9,Romania,11
